#### Import des librairies

In [1]:
import pandas as pd
from tqdm import tqdm
import numpy as np

In [3]:

ENTREPOT_PATH = '~/Bureau/utils/data/'
DIRODUR_FILES_PATH = '~/Bureau/projets/DIRODUR/magasin/'
df = {}

#### Import des données

In [4]:
# ----------------------------- #
# IMPORT DES DONNÉES DATAGROSYST#
# ----------------------------- #


def import_df(df_name, path_data, sep, index_col=None):
    df[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, index_col=index_col, low_memory=False).replace({'\r\n': '\n'}, regex=True)

def import_dfs(df_names, path_data, sep = ',', index_col=None, verbose=False):
    for df_name in tqdm(df_names) : 
        if(verbose) :
            print(" - ", df_name)
        import_df(df_name, path_data, sep, index_col=index_col)

tables_with_id = [
    'espece', 'culture', 'composant_culture', 'noeuds_realise', 'zone', 'sdc', 'noeuds_synthetise_restructure', 'noeuds_synthetise',
    'connection_synthetise'
]

tables_without_id = [
    'typologie_assol_can_realise', 'typologie_can_culture', 'itk_realise_agrege', 'itk_synthetise_agrege'
]

# import des données de l'entrepôt avec la colonne 'id' en index 
import_dfs(tables_with_id, ENTREPOT_PATH, sep = ',', index_col='id', verbose=False)

# import des données du magasin
import_dfs(tables_without_id, ENTREPOT_PATH, sep = ',', verbose=False)

  0%|          | 0/9 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:04<00:00,  1.19s/it]


In [5]:
studied_ids = [
    # Porte graine
    'fr.inra.agrosyst.api.entities.GrowingSystem_88bd9f2a-5604-4171-9b00-26af988f5441',
    'fr.inra.agrosyst.api.entities.GrowingSystem_ac66c1bc-1a96-492b-bf76-64087319872d',
    'fr.inra.agrosyst.api.entities.GrowingSystem_c7d5fc9e-4560-4a2c-8b1e-7af04779b820',
    #Divers
    'fr.inra.agrosyst.api.entities.GrowingSystem_8f9c8256-bb5f-46f8-89bd-9c1d89a5e4c1',
    'fr.inra.agrosyst.api.entities.GrowingSystem_219d626f-9be8-431a-9458-d491820d1dc8',
    'fr.inra.agrosyst.api.entities.GrowingSystem_f4466535-63ba-4afd-a63e-a1a4afad4965',
    'fr.inra.agrosyst.api.entities.GrowingSystem_97dd5d5c-e75c-49c2-bdc2-dcddf4c53244',
    'fr.inra.agrosyst.api.entities.GrowingSystem_240b8c12-8982-47be-8313-5bcccfc6ec22',
    'fr.inra.agrosyst.api.entities.GrowingSystem_8b2b677f-572e-4f04-abbf-ca56da468416',
    'fr.inra.agrosyst.api.entities.GrowingSystem_751c6351-04cd-44e3-94f1-7d84c66b1622'
]

In [6]:

def get_percent_each_typo_culture(cgrp, freq_column='frequence', normalize=True):
    '''
    Permet de calculer le pourcentage de chaque typologie de culture dans un groupe de données.
    Ce groupe de données est généralement un groupe de données de rotation pour le synthétisé ou un sdc pour le réalisé.

    Args:
        cgrp (pd.DataFrame):
            DataFrame de données de la rotation pour le synthétisé ou du sdc pour le réalisé.
        freq_column (str):
            Nom de la colonne de fréquence à utiliser pour les calculs. Par défaut 'frequence' pour le synthétisé.
            Peut être 'surface_ponderee' ou 'surface' pour le réalisé.
        normalize (bool):
            Si True, retour en pourcentage
            Si False, retour en ha
    Returns:
        dict: Dictionnaire des pourcentages de chaque typologie de culture dans le groupe de données,
        trié par pourcentage décroissant. Clé = typologie, valeur = pourcentage (float).
        Si aucune fréquence renseignée, retourne {'erreur': 'aucune '+freq_column+' renseignée'}
        Si la somme des surfaces est nulle, retourne {'erreur': freq_column+' nulle renseignée'}
        Si la somme des surfaces est inférieure à 0.001, retourne {'erreur': freq_column+' totale < 0.001'}
    '''
    cgrp = cgrp.copy()
    cgrp['typocan_culture_sans_compagne_corrige'] = cgrp['typocan_culture_sans_compagne_corrige'].fillna('NoTypoC')

    if pd.isna(cgrp[freq_column]).all():
        return {'erreur': 'aucune ' + freq_column + ' renseignée'}

    if freq_column in {'surface_ponderee', 'surface'}:
        surf_sum = cgrp[freq_column].sum()
        if surf_sum == 0:
            return {'erreur': freq_column + ' nulle renseignée'}
        if surf_sum < 0.001:
            return {'erreur': freq_column + ' totale < 0.001'}

    percentages = {}
    for x in cgrp['typocan_culture_sans_compagne_corrige'].unique():
        typoc_sum = cgrp.loc[cgrp['typocan_culture_sans_compagne_corrige'] == x, freq_column].sum()
        if freq_column == 'frequence':
            typoc_sum = typoc_sum * 100
        elif freq_column in {'surface_ponderee', 'surface'}:
            if(normalize):
                typoc_sum = (typoc_sum / surf_sum) * 100
            else: 
                typoc_sum = typoc_sum
        percentages[x] = round(typoc_sum, 1)

    # Trier par pourcentage décroissant
    percentages = dict(
        sorted(percentages.items(), key=lambda item: item[1], reverse=True)
    )

    return percentages

In [7]:
def get_surface_typo_culture_sdc_realise_outils_tableau_de_bord_can(donnees):
    """
        Retourne pour chaque système de culture en realise, la surface par typologie de culture (une colonne par typologie de culture)

        > Attention, toutes les culture déclarées "porte-graines" ne doivent pas être décomptées dans les autres typologies de culture. 
    
        Tables nécessaires :
        - noeuds_realise
        - itk_realise_agrege
        - zone
        - typologie_can_culture
    
    """
    left = df['noeuds_realise'].reset_index()
    right = df['itk_realise_agrege'][['itk_id',  'sdc_id']]
    df['noeuds_realise_extanded'] = pd.merge(left, right, left_on='id', right_on='itk_id', how='left').set_index('id')

    # ajout du poids du noeud, c'est à dire la surface en réalisé
    left = df['noeuds_realise_extanded']
    right = df['zone'][['surface']]
    df['noeuds_realise_extanded'] = pd.merge(left, right, left_on='zone_id', right_index=True, how='left')

    # On créé une nouvelle colonne "typocan_culture_corrige" qui réaffecte les culture porte graine à une typologie dédiée (volonté Cellule Ref)
    df['typologie_can_culture']['typocan_culture_sans_compagne_corrige'] = df['typologie_can_culture']['typocan_culture']
    df['typologie_can_culture'].loc[
        df['typologie_can_culture']['typo_cpg'].isin(
            ['Cultures porte graines', 'Cultures porte graines et autres destinations']
        ), 'typocan_culture_sans_compagne_corrige'
    ] = 'Porte graine'

    left = df['noeuds_realise_extanded']
    right = df['typologie_can_culture'].set_index('culture_id')[['typocan_culture_sans_compagne_corrige']]
    df['noeuds_realise_extanded'] = pd.merge(left, right, left_on='culture_id', right_index=True, how='left')

    result = df['noeuds_realise_extanded'].groupby('sdc_id').apply(
        lambda g: get_percent_each_typo_culture(g, freq_column='surface', normalize=False)
    )

    result_df = result.apply(pd.Series).fillna(0).rename(columns={
        'Betterave' : 'surface_betterave', 
        'Céréales à paille printemps' : 'surface_cereale_a_paille_printemps',
        'Céréales à paille hiver' : 'surface_cereale_a_paille_hiver',
        'Colza' : 'surface_colza',
        'Légume' : 'surface_legume', # Attention, légume plein champs
        'Lin' : 'surface_lin', # Attention, Lin fibre
        'Maïs' : 'surface_mais', # ATtention, Maïs Sorgho
        'Mélange fourrager' : 'surface_melange_fourrager',
        'Oléagineux (hors Colza et Tournesol)' : 'surface_oleagineux', # Attention, Olea
        'Pomme de terre' : 'surface_pomme_de_terre', 
        'Porte graine': 'surface_porte_graine', 
        'Prairie temporaire' : 'surface_prairie_temporaire',
        'Protéagineux' : 'surface_proteagineux',
        'Tournesol' : 'surface_tournesol',
        'Autre' : 'surface_autre',
        'Pommier' : 'surface_pommier', 
        'Vigne' : 'surface_vigne', 
        'Plante aromatique ou médicinale' : 'surface_plante_aromatique_ou_medicinale', 
        'Fraisier' : 'surface_fraisier', 
        'NoInput-sp' : 'surface_NoInput-sp', 
        'Prunier' : 'surface_prunier',
        'Culture ornementale' : 'surface_culture_ornementale',
        'NoTypoC' : 'surface_NoTypoC',
        'erreur' : 'surface_erreur;', 
        'Pêcher' : 'surface_pecher', 
        'Litchi': 'surface_litchi', 
        'Ananas' : 'surface_ananas',
        'Bananier' : 'surface_bananier', 
        'Attier' : 'surface_attier', 
        'Fruit de la passion' : 'surface_fruit_de_la_passion', 
        'Cerisier' : 'surface_cerisier', 
        'Poirier' : 'surface_poirier',
        'Papayer' : 'surface_papayer', 
        'Cacaoyer' : 'surface_cacaoyer', 
        'Framboisier' : 'surface_framboisier', 
        'Petits fruits' : 'surface_petits_fruits',
        'Arbre à pain' : 'surface_arbre_a_pain',
        'Figuier' : 'surface_figuier',
        'Sapin' : 'surface_sapin', 
        'Canne à sucre' : 'surface_canne_a_sucre',
        'Groseiller' : 'surface_groseiller', 
        'Grenadille' : 'surface_grenadille',
        'Manguier' : 'surface_manguier', 
        'Noyer' : 'surface_noyer', 
        'Cassissier' : 'surface_casssissier',
        'Citronnier' : 'surface_citronnier'
    })

    return result_df

In [8]:
result_df = get_surface_typo_culture_sdc_realise_outils_tableau_de_bord_can(df)

/tmp/ipykernel_16316/50581609.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = df['noeuds_realise_extanded'].groupby('sdc_id').apply(


In [9]:
left = df['noeuds_realise']
right = df['itk_realise_agrege'].set_index('itk_id')[['sdc_id']]
df['noeuds_realise_extanded'] = pd.merge(left, right, left_index=True, right_index=True, how='left')

In [10]:
df['noeuds_realise_extanded_test']= df['noeuds_realise_extanded'].loc[
    df['noeuds_realise_extanded']['sdc_id'].isin(studied_ids)
]

df['noeuds_realise_test']= df['noeuds_realise'].loc[
    df['noeuds_realise'].index.isin(df['noeuds_realise_extanded_test'].index)
]
df['itk_realise_agrege_test'] = df['itk_realise_agrege'].loc[
    df['itk_realise_agrege']['itk_id'].isin(df['noeuds_realise_test'].index)
]
df['zone_test'] = df['zone'].loc[
    df['zone'].index.isin(df['noeuds_realise_test']['zone_id'])
]
df['typologie_can_culture_test'] = df['typologie_can_culture'].loc[
    df['typologie_can_culture']['culture_id'].isin(df['noeuds_realise_test']['culture_id'])
]

In [11]:
# export 
path='./'
df['noeuds_realise_test'].to_csv(path+'noeuds_realise.csv')
df['itk_realise_agrege_test'].to_csv(path+'itk_realise_agrege.csv')
df['zone_test'].to_csv(path+'zone'+'.csv')
df['typologie_can_culture_test'].to_csv(path+'typologie_can_culture'+'.csv')